# CEE6501 — Coding Assignment, Week 5

**Assigned:** 02/13/2026 (Week 5)  
**Due:** 02/23/2026

**Canvas Submission Link:**  <https://gatech.instructure.com/courses/517856/assignments/2320760>

---

## Logistics

### 💻 Assignment Format

This is a **coding assignment**.

- Complete the assignment by **executing and completing all tasks in the notebook cells below**
- The notebook should be run and completed in **Google Colab**
- Your submission **must be a link to a functioning Google Colab notebook**

You may use any local tools (VS Code, JupyterLab, etc.) while working, but the **final submitted work must run correctly in Colab**.


### 📤 Submission Instructions

- Submit **one link** to your Google Colab notebook on Canvas
- Ensure that:
  - All cells run **top-to-bottom without errors**
  - All required outputs are visible
  - The notebook reflects your final answers

### ✅ Checklist Before Submitting

- [ ] All notebook cells completed
- [ ] Code runs without errors from a fresh runtime
- [ ] Outputs and plots are clearly visible
- [ ] Colab link opens and runs correctly
- [ ] Correct notebook submitted on Canvas

### 🤝 Collaboration / AI tools
You may discuss concepts with classmates and you may use AI tools to help you learn,
but **your submitted code must be written by you and you must understand it**.
If you used outside help, add a short note in the final reflection cell.

---

## --- Google Colab environment setup ---

The cell below only needs to run when the notebook is opened in Google Colab.

This code will not affect code execution locally in VS-code + conda environment.

Google Colab starts each session with its own **preloaded versions** of common Python (currently 3.12.12) and Python packages (NumPy, SciPy, etc.).  
If we install different package versions once loaded, Python cannot switch to them while it is already running.

### What will happen
When you run the setup cell below in Google Colab:

1. The required package versions are installed
2. The runtime is **automatically restarted** so the new versions can be loaded  
3. You may see the message **“Your session crashed for an unknown reason.”**  
   → This is expected and normal

After the restart, rerun the notebook and check the **version check cell** to confirm package versions are correct.

### Runtime menu notes
- **Runtime → Restart session**  
  Restarts Python but keeps installed packages and saved files

- **Runtime → Disconnect and delete runtime**  
  Resets Colab completely to its default environment (packages will need to be reinstalled)

In [163]:
# ============================================================
# Google Colab environment setup (pinned versions)
# ============================================================

import sys
import os
import subprocess

if "google.colab" in sys.modules:
    print("Running in Google Colab")
    print("Python version:", sys.version.split()[0])

    # ---- Required package versions --------------------------
    requirements = {
        "numpy": "2.4.0",
        "scipy": "1.16.3",
        "matplotlib": "3.10.8",
        "pandas": "2.3.3",
        "plotly": "6.5.2"
    }

    # ---- Check currently loaded versions --------------------
    restart_needed = False

    for pkg, required_version in requirements.items():
        try:
            module = __import__(pkg)
            installed_version = module.__version__
        except Exception:
            installed_version = None

        print(f"{pkg}: {installed_version} (required: {required_version})")

        if installed_version != required_version:
            restart_needed = True

    # ---- Install if needed ----------------------------------
    if restart_needed:
        print("\nInstalling pinned package versions...")

        pip_args = [
            f"{pkg}=={ver}" for pkg, ver in requirements.items()
        ]

        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *pip_args]
        )

        print("Installation complete.")
        print("Restarting runtime to load correct packages...")

        # This will appear as a "crash" in Colab — expected behavior
        os.kill(os.getpid(), 9)

    else:
        print("\nAll required package versions already installed.")

else:
    print("Not running in Google Colab — setup skipped.")
    print("Python version:", sys.version.split()[0])

Not running in Google Colab — setup skipped.
Python version: 3.12.12


In [164]:
# --- Version check ---
import numpy
import scipy
import matplotlib
import pandas
import plotly

print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)
print("pandas:", pandas.__version__)
print("plotly:", plotly.__version__)

numpy: 2.4.0
scipy: 1.16.3
matplotlib: 3.10.8
pandas: 2.3.3
plotly: 6.5.2


---
---

## Imports

Run this cell once before starting the assignment.

In [165]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go

np.set_printoptions(precision=3, suppress=True)


---

## GIVEN CODE (You will need to modify):

## Parsing & Helper Functions

In [166]:
def restrained_dofs_1based(nodes_restrained, node_dofs_1based):
    """Return sorted list of restrained DOFs (1-based) from node restraints."""
    dof_restrained = []

    for node, restraints in nodes_restrained.items():
        ux_dof, uy_dof, uz_dof = node_dofs_1based(node)
        if "ux" in restraints:
            dof_restrained.append(ux_dof)
        if "uy" in restraints:
            dof_restrained.append(uy_dof)
        if "uz" in restraints:
            dof_restrained.append(uz_dof)

    return sorted(dof_restrained)

In [167]:
def loaded_dofs_1based(nodes_loaded, node_dofs_1based):
    """Return DOF→load mapping (1-based) from node loads."""
    dof_loaded = {}

    for node, (Fx, Fy, Fz) in nodes_loaded.items():
        ux_dof, uy_dof, uz_dof = node_dofs_1based(node)
        if Fx != 0.0:
            dof_loaded[ux_dof] = dof_loaded.get(ux_dof, 0.0) + Fx
        if Fy != 0.0:
            dof_loaded[uy_dof] = dof_loaded.get(uy_dof, 0.0) + Fy
        if Fz != 0.0:
            dof_loaded[uz_dof] = dof_loaded.get(uz_dof, 0.0) + Fz

    return dof_loaded


In [168]:
def node_dofs_1based(node_id):
    """Return engineering DOF numbers (1-based): [ux_dof, uy_dof, uz_dof]."""
    return [3 * node_id - 2, 3 * node_id - 1, 3 * node_id]

## Element Level

In [169]:
def element_lmnL(xyz_i, xyz_j):
    xyz_i = np.asarray(xyz_i, dtype=float)
    xyz_j = np.asarray(xyz_j, dtype=float)

    dx = xyz_j[0] - xyz_i[0]
    dy = xyz_j[1] - xyz_i[1]
    dz = xyz_j[2] - xyz_i[2]
    L = float(np.sqrt(dx*dx + dy*dy + dz*dz))

    l = dx / L
    m = dy / L
    n = dz / L
    return float(l), float(m), float(n), float(L)

In [170]:
def build_elements_lmnL(elements, nodes):
    """Return dict mapping element_id -> (l, m, n, L)."""
    elements_lmnL = {}

    for e_id, (i, j, E_e, A_e) in elements.items():
        l, m, n, L = element_lmnL(nodes[i], nodes[j])
        elements_lmnL[e_id] = (l, m, n, L)

    return elements_lmnL

## Global Stiffness and Force

In [171]:
def initialize_global_stiffness(nodes):
    """Return zero-initialized global stiffness matrix."""
    ndof_total = 3 * len(nodes)
    return np.zeros((ndof_total, ndof_total), dtype=float)

In [172]:
def truss_element_kg(E, A, l, m, n, L):
    """Return 6x6 global stiffness matrix for a 3D truss element."""
    factor = (E * A) / L

    l2 = l * l
    m2 = m * m
    n2 = n * n
    lm = l * m
    ln = l * n
    mn = m * n

    kg = factor * np.array(
        [
            [ l2,  lm,  ln, -l2, -lm, -ln],
            [ lm,  m2,  mn, -lm, -m2, -mn],
            [ ln,  mn,  n2, -ln, -mn, -n2],
            [-l2, -lm, -ln,  l2,  lm,  ln],
            [-lm, -m2, -mn,  lm,  m2,  mn],
            [-ln, -mn, -n2,  ln,  mn,  n2],
        ],
        dtype=float,
    )
    return kg

In [173]:
def element_dof_map_1based(i_node, j_node):
    """Return the 6 global DOF indices (1-based) for element (i, j).

    Order matches the 6x6 element stiffness matrix:
    [u_ix, u_iy, u_iz, u_jx, u_jy, u_jz]
    """
    # Engineering DOF numbers (1-based)
    dofs_i_1 = [3 * i_node - 2, 3 * i_node - 1, 3 * i_node]
    dofs_j_1 = [3 * j_node - 2, 3 * j_node - 1, 3 * j_node]
    dofs_1based = dofs_i_1 + dofs_j_1
    return dofs_1based

In [174]:
def assemble_global_stiffness(elements, nodes, elements_lmnL, print_toggle):
    """Assemble and return the global stiffness matrix K (dense) for a 3D truss."""

    K = initialize_global_stiffness(nodes)

    for e_id, (i, j, E_e, A_e) in elements.items():
        l, m, n, L = elements_lmnL[e_id]
        ke = truss_element_kg(E_e, A_e, l, m, n, L)
        
        dof_map = element_dof_map_1based(i, j)

        # Scatter-add ke into K. Must be 0-based indexing, hence -1
        for a in range(6):
            A = dof_map[a] - 1
            for b in range(6):
                B = dof_map[b] - 1
                K[A, B] += ke[a, b]

        # Optional: show progress while learning/debugging
        if print_toggle:
            print_matrix_scaled(ke)
            print(f"Assembled element {e_id}: nodes ({i},{j}) -> DOFs {dof_map}")
            print_matrix_scaled(K)
            print("-" * 70)

    return K

In [175]:
def assemble_global_force_vector(nodes, dof_loaded_1based):
    """Return global force vector assembled from 1-based DOF loads."""
    ndof_total = 3 * len(nodes)
    f_global = np.zeros(ndof_total, dtype=float)

    for dof_1based, value in dof_loaded_1based.items():
        f_global[dof_1based - 1] = value  # convert to 0-based index

    return f_global

## Partition

In [176]:
def partition_system(K, f, dof_restrained_1based):
    ndof = K.shape[0]

    # Convert restrained DOFs to 0-based
    restrained_dofs = sorted(d - 1 for d in dof_restrained_1based)

    # Free DOFs
    free_dofs = [i for i in range(ndof) if i not in restrained_dofs]

    # Partition stiffness matrix
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fr = K[np.ix_(free_dofs, restrained_dofs)]
    K_rf = K[np.ix_(restrained_dofs, free_dofs)]
    K_rr = K[np.ix_(restrained_dofs, restrained_dofs)]

    # Partition force vector
    f_f = f[free_dofs]
    f_r = f[restrained_dofs]

    return K_ff, K_fr, K_rf, K_rr, f_f, f_r, free_dofs, restrained_dofs


## Solve Displacements

In [177]:
def solve_free_displacements(K_ff, K_fr, f_f, u_r=None):
    if u_r is None:
        u_r = np.zeros(K_fr.shape[1])

    rhs = f_f - K_fr @ u_r
    u_f = np.linalg.solve(K_ff, rhs)

    return u_f


## Solve Support Forces

In [178]:
def solve_support_forces(K_rf, K_rr, u_f, u_r=None):
    if u_r is None:
        u_r = np.zeros(K_rr.shape[0])

    F_r = K_rf @ u_f + K_rr @ u_r
    return F_r


## Backward Pass

In [179]:
def assemble_global_displacements(u_f, free_dofs, restrained_dofs, u_r=None):
    """
    Assemble the full global displacement vector u from partitioned results.
    """
    ndof_total = len(free_dofs) + len(restrained_dofs)
    u_global = np.zeros(ndof_total)

    if u_r is None:
        u_r = np.zeros(len(restrained_dofs))

    u_global[free_dofs] = u_f
    u_global[restrained_dofs] = u_r

    return u_global


In [180]:
def assemble_global_forces(f_f, F_r, free_dofs, restrained_dofs):
    """
    Assemble the full global force vector f from applied loads and reactions.
    """
    ndof_total = len(free_dofs) + len(restrained_dofs)
    f_global = np.zeros(ndof_total)

    f_global[free_dofs] = f_f
    f_global[restrained_dofs] = F_r

    return f_global


In [181]:
def extract_element_displacements(u_global, i_node, j_node):
    """
    Extract the 6x1 element global displacement vector u_e.
    Order: [u_ix, u_iy, u_iz, u_jx, u_jy, u_jz]
    """
    dofs_1based = [3 * i_node - 2, 3 * i_node - 1,3 * i_node,
                   3 * j_node - 2, 3 * j_node - 1,3 * j_node]
    idx = [d - 1 for d in dofs_1based]  # convert to 0-based
    return u_global[idx]

In [182]:
def truss_transformation_matrix(l, m, n, a=None):
    
    ex = np.array([l, m, n], dtype=float)
    ex_norm = np.linalg.norm(ex)
    if ex_norm < 1e-12:
        raise ValueError("Direction cosines (l,m,n) are invalid (near zero vector).")
    ex = ex / ex_norm
    l, m, n = ex.tolist()

    # --- choose reference vector a (if not provided) ---
    if a is None:
        # default: use global Z unless too parallel; otherwise use global Y
        a = np.array([0.0, 0.0, 1.0]) if abs(n) < 0.9 else np.array([0.0, 1.0, 0.0])
    else:
        a = np.array(a, dtype=float)
        a_norm = np.linalg.norm(a)
        if a_norm < 1e-12:
            raise ValueError("Reference vector a is invalid (near zero).")
        a = a / a_norm

    # --- ey' = normalize(a x ex') ---
    ey = np.cross(a, ex)
    ey_norm = np.linalg.norm(ey)
    if ey_norm < 1e-12:
        # a is (almost) parallel to ex -> pick another a automatically
        a2 = np.array([0.0, 1.0, 0.0]) if abs(n) < 0.9 else np.array([1.0, 0.0, 0.0])
        ey = np.cross(a2, ex)
        ey_norm = np.linalg.norm(ey)
        if ey_norm < 1e-12:
            raise ValueError("Failed to build transverse axes; choose a different reference vector a.")
    ey = ey / ey_norm

    # --- ez' = ex' x ey' ---
    ez = np.cross(ex, ey)

    ly, my, ny = ey.tolist()
    lz, mz, nz = ez.tolist()

    return np.array(
        [
            [l, m, n, 0, 0, 0],
            [ly, my, ny, 0, 0, 0],
            [lz, mz, nz, 0, 0, 0],
            [0, 0, 0, l, m, n],
            [0, 0, 0, ly, my, ny],
            [0, 0, 0, lz, mz, nz],
        ],
        dtype=float,
    )

def compute_local_displacements(l, m, n, u_e):
    T = truss_transformation_matrix(l, m, n, a=None)
    return T @ u_e



In [183]:
def truss_local_stiffness(E, A, L):
    return (E * A / L) * np.array(
        [
            [ 1, 0, 0, -1, 0, 0],
            [ 0, 0, 0, 0, 0, 0],
            [ 0, 0, 0, 0, 0, 0],
            [-1, 0, 0, 1, 0, 0],
            [ 0, 0, 0, 0, 0, 0],
            [ 0, 0, 0, 0, 0, 0],
        ],
        dtype=float,
    )

In [184]:
def compute_local_end_forces(E, A, L, u_local):
    k_local = truss_local_stiffness(E, A, L)
    return k_local @ u_local

In [185]:
def compute_axial_force_and_stress(E, A, L, u_local):
    # axial displacements are the local DOFs along the member axis
    u_i_axial = u_local[0]
    u_j_axial = u_local[3]

    N = (E * A / L) * (u_j_axial - u_i_axial)
    sigma = N / A
    return N, sigma

In [186]:
def local_to_global_forces(l, m, n, f_local):
    T = truss_transformation_matrix(l, m, n)
    return T.T @ f_local

## Post-Processing & Output

In [187]:
def plot_truss_deformation(nodes, elements, u_global, scale=1.0):
    """
    Plot original (black) and deformed (red) truss geometry.
    """
    fig = go.Figure()

    first = True

    for e_id, (i, j, *_ ) in elements.items():
        xi, yi, zi = nodes[i]
        xj, yj, zj = nodes[j]

        ui = u_global[3*(i-1):3*(i-1)+3]
        uj = u_global[3*(j-1):3*(j-1)+3]

        # original
        fig.add_trace(go.Scatter3d(
            x=[xi, xj], y=[yi, yj], z=[zi, zj],
            mode="lines",
            name="Original" if first else None,
            showlegend=first,
            line=dict(width=4)
        ))

        # deformed
        fig.add_trace(go.Scatter3d(
            x=[xi + scale*ui[0], xj + scale*uj[0]],
            y=[yi + scale*ui[1], yj + scale*uj[1]],
            z=[zi + scale*ui[2], zj + scale*uj[2]],
            mode="lines",
            name="Deformed" if first else None,
            showlegend=first,
            line=dict(width=4)
        ))

        first = False

    fig.update_layout(
        title=f"Original and deformed truss, scale={scale}",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode="data"  # equal
        )
    )

    fig.show()
    return fig


In [188]:
def recover_element_results(elements, elements_lmnL, u_global):
    results = {}

    for e_id, (i, j, E_e, A_e) in elements.items():
        l, m, n, L = elements_lmnL[e_id]

        u_e = extract_element_displacements(u_global, i, j)
        u_local = compute_local_displacements(l, m, n, u_e)
        f_local = compute_local_end_forces(E_e, A_e, L, u_local)
        N, sigma = compute_axial_force_and_stress(E_e, A_e, L, u_local)

        results[e_id] = {
            "u_e": u_e,
            "u_local": u_local,
            "f_local": f_local,
            "N": N,          # kN
            "sigma": sigma,  # GPa
        }

    return results

In [189]:
def build_element_results_dataframe(elements, elements_lmnL, results):
    """Return pandas DataFrame of element-level results."""
    rows = []

    for e_id in sorted(elements.keys()):
        i, j, _, _ = elements[e_id]
        _, _, _, L = elements_lmnL[e_id]
        r = results[e_id]

        row = {
            "ele": e_id,
            "i": i,
            "j": j,
            "L (mm)": round(L, 1),
            "N (kN)": round(r["N"], 1),
            "sigma (MPa)": round(r["sigma"] * 1000, 1),
        }

        # global displacements u_e
        row.update({f"u_{k+1} (mm)": round(r["u_e"][k], 1) for k in range(6)})
        # local displacements u'
        row.update({f"u_{k+1}' (mm)": round(r["u_local"][k], 1) for k in range(6)})
        # local end forces f'
        row.update({f"f_{k+1}' (kN)": round(r["f_local"][k], 1) for k in range(6)})

        rows.append(row)

    return pd.DataFrame(rows)


In [190]:
def display_compact(df):
    return (
        df.style
        .format({
            col: "{:.1f}"
            for col in df.columns
            if any(key in col for key in ["(mm)", "(kN)", "(MPa)", "L"])
        })
        .set_properties(**{
            "font-size": "9pt",
            "padding": "2px",
            "white-space": "nowrap",
        })
        .set_table_styles([
            {"selector": "th", "props": [("font-size", "9pt")]},
        ])
    )

In [191]:
def print_matrix_scaled(K, scale=1, decimals=2, col_width=7):
    """
    Print K/scale row-by-row, compact, with DOF labels.
    """
    fmt = f"{{:{col_width}.{decimals}f}}"
    print(f"K = {scale:.0e} ×")
    for i, row in enumerate(K, start=1):
        row_scaled = row / scale
        row_str = " ".join(fmt.format(val) for val in row_scaled)
        print(f"{i:2d} | {row_str}")

In [192]:
def build_global_load_vector(n_dof, dof_loaded_1based):
    """dof_loaded_1based: {dof(1-based): value}  ->  F (0-based numpy vector)"""
    F = np.zeros(n_dof, dtype=float)
    for dof1, val in dof_loaded_1based.items():
        F[dof1 - 1] += val
    return F

In [193]:
def build_reaction_df(nodes_restrained, node_dofs_1based,
                      K_global, u_global, dof_loaded_1based,
                      decimals=1):
    """
    Return a DataFrame with support reactions at restrained nodes.
    Units: same as your loads (e.g., kN).
    """
    n_dof = K_global.shape[0]
    F = build_global_load_vector(n_dof, dof_loaded_1based)

    R = K_global @ u_global - F

    rows = []
    for node in sorted(nodes_restrained.keys()):
        ux, uy, uz = node_dofs_1based(node)

        Rx = R[ux - 1]
        Ry = R[uy - 1]
        Rz = R[uz - 1]

        rows.append({
            "node": node,
            "Rx (kN)": round(Rx, decimals),
            "Ry (kN)": round(Ry, decimals),
            "Rz (kN)": round(Rz, decimals)
        })

    return pd.DataFrame(rows)

In [194]:
def build_node_displacement_df(nodes, node_dofs_1based, u_global):
    """
    nodes: {node_id: (x,y,z)}
    node_dofs_1based(node) -> (ux_dof, uy_dof, uz_dof)  1-based
    u_global: full displacement vector
    """
    rows = []
    for n in sorted(nodes.keys()):
        coord = nodes[n]
        dofs = node_dofs_1based(n)

        ux = float(u_global[dofs[0]-1])
        uy = float(u_global[dofs[1]-1])
        uz = float(u_global[dofs[2]-1])

        umag = float(np.sqrt(ux**2 + uy**2 + uz**2))

        row = {"node": n, "x": coord[0], "y": coord[1], "ux (mm)": ux, "uy (mm)": uy, "uz (mm)": uz, "|u| (mm)": umag}

        rows.append(row)

    df = pd.DataFrame(rows)

    for c in df.columns:
        df[c] = df[c].astype(float).round(1)

    return df


In [195]:
def build_max_displacement_summary(df_disp):
    """
    Extract the maximum displacement (modulus length) and the positions of the maximum absolute values for each component from the node displacement table.
    """
    cols = df_disp.columns

    i_mag = df_disp["|u| (mm)"].astype(float).idxmax()
    r_mag = df_disp.loc[i_mag]

    summary_rows = [{
        "metric": "max |u|",
        "node": int(r_mag["node"]),
        "value (mm)": float(r_mag["|u| (mm)"]),
        "ux (mm)": float(r_mag["ux (mm)"]),
        "uy (mm)": float(r_mag["uy (mm)"]),
        "uz (mm)": float(r_mag["uz (mm)"])
    }]

    for comp in ["ux (mm)", "uy (mm)", "uz (mm)"]:
        if comp not in cols:
            continue
        i_c = df_disp[comp].astype(float).abs().idxmax()
        r_c = df_disp.loc[i_c]
        summary_rows.append({
            "metric": f"max |{comp.split()[0]}|",
            "node": int(r_c["node"]),
            "value (mm)": float(r_c[comp]),
            "ux (mm)": float(r_c["ux (mm)"]),
            "uy (mm)": float(r_c["uy (mm)"]),
            "uz (mm)": float(r_c["uz (mm)"])
        })

    return pd.DataFrame(summary_rows)

---
---

## Question 3 — Extending the Truss Analysis Code to 3D (20 points)

Extend the functions defined at the beginning of this workbook so that they operate in **three dimensions**. This requires:

- Updating the number of degrees of freedom per node from **2 → 3**
- Modifying all related indexing, assembly, and bookkeeping to be consistent with 3D kinematics
- Use the 3D local/global truss element formulation presented in class
- Applying an appropriate set of **support restraints** for a 3D truss system (including out-of-plane constraints)

Verify that the **statically determinate truss** analyzed in the written assignment produces **consistent and correct results** when solved using your 3D computational implementation. You should get the same answer across 2D and 3D implementation if you are doing things correctly.

### Implementation Guidance

As an example, in the function below, extending the formulation to 3D requires changing the total number of degrees of freedom from `2 * len(nodes)` to `3 * len(nodes)`. Similar changes will be required throughout your code so that it correctly assembles, solves, and visualizes a **3D structure**. Use a **3D axis** when plotting with `matplotlib`, or an even better option is to use `plotly` to plot dynamic 3D plots.

```python
def assemble_global_force_vector(nodes, dof_loaded_1based):
    """Return global force vector assembled from 1-based DOF loads."""
    ndof_total = 2 * len(nodes)
    f_global = np.zeros(ndof_total, dtype=float)

    for dof_1based, value in dof_loaded_1based.items():
        f_global[dof_1based - 1] = value  # convert to 0-based index

    return f_global


### Q3.1. Problem Setup
Define all model inputs (nodes, elements, supports, loads) using the same data structures and conventions as we did in lecture but extended to 3D.

Present:
- Node coordinates
- Element connectivity and properties
- Applied loads and boundary conditions

In [196]:
# -------------------------
# Nodes: (x,y,z)
# -------------------------

import math

B = 3000.0

nodes = {
    # station 1
    1:  (0.0,      0.0,              0.0),
    2:  (0.0,      0.0,              B),

    # station 2
    3:  (4500.0,   1500.0*math.sqrt(3), 0.0),
    4:  (4500.0,   1500.0*math.sqrt(3), B),

    # station 3
    5:  (6000.0,   0.0,              0.0),
    6:  (6000.0,   0.0,              B),

    # station 4
    7:  (9000.0,   3000.0*math.sqrt(3), 0.0),
    8:  (9000.0,   3000.0*math.sqrt(3), B),

    # station 5
    9:  (12000.0,  0.0,              0.0),
    10: (12000.0,  0.0,              B),

    # station 6
    11: (13500.0,  1500.0*math.sqrt(3), 0.0),
    12: (13500.0,  1500.0*math.sqrt(3), B),

    # station 7
    13: (18000.0,  0.0,              0.0),
    14: (18000.0,  0.0,              B)
}

In [197]:
E = 70.0   # GPa
A = 4000.0    # mm^2

# -------------------------
# Elements: (i, j, E, A)
# -------------------------
elements = {
    1:  (1,  5,  E, A),   # (1,3)
    2:  (1,  3,  E, A),   # (1,2)
    3:  (3,  5,  E, A),   # (2,3)
    4:  (3,  7,  E, A),   # (2,4)
    5:  (5,  7,  E, A),   # (3,4)
    6:  (5,  9,  E, A),   # (3,5)
    7:  (7,  9,  E, A),   # (4,5)
    8:  (7,  11, E, A),   # (4,6)
    9:  (9,  11, E, A),   # (5,6)
    10: (9,  13, E, A),   # (5,7)
    11: (11, 13, E, A),   # (6,7)

    # =========================
    # Longitudinal truss (Row z=B)
    # =========================
    12: (2,  6,  E, A),   # (1,3)
    13: (2,  4,  E, A),   # (1,2)
    14: (4,  6,  E, A),   # (2,3)
    15: (4,  8,  E, A),   # (2,4)
    16: (6,  8,  E, A),   # (3,4)
    17: (6,  10, E, A),   # (3,5)
    18: (8,  10, E, A),   # (4,5)
    19: (8,  12, E, A),   # (4,6)
    20: (10, 12, E, A),   # (5,6)
    21: (10, 14, E, A),   # (5,7)
    22: (12, 14, E, A),   # (6,7)

    # =========================
    # Transverse bracing
    # =========================
    23: (1,  2,  E, A),
    24: (3,  4,  E, A),
    25: (5,  6,  E, A),
    26: (7,  8,  E, A),
    27: (9,  10, E, A),
    28: (11, 12, E, A),
    29: (13, 14, E, A),

    # =========================
    # Transverse bracin X
    # =========================
    # station1-2
    30: (1,  4,  E, A),
    31: (2,  3,  E, A),

    # station1-3
    32: (1,  6,  E, A),
    33: (2,  5,  E, A),

    # station2-4
    34: (3,  8,  E, A),
    35: (4,  7,  E, A),

    # station3-5
    36: (5,  10,  E, A),
    37: (6,  9,  E, A),

    # station4-6
    38: (7,  12, E, A),
    39: (8,  11,  E, A),

    # station5-7
    40: (9, 14, E, A),
    41: (10, 13, E, A),

    # station6-7
    42: (11, 14, E, A),
    43: (12, 13, E, A)
}

elements_lmnL = build_elements_lmnL(elements, nodes)

In [198]:
# -------------------------
# Supports / restraints
# -------------------------
nodes_restrained = {
    1:  ["ux", "uy", "uz"],   # pin
    2:  ["ux", "uy", "uz"],
    13: ["uy"],         # y-roller
    14: ["uy"]
}

# -------------------------
# Applied nodal loads (Fx, Fy, Fz)
# -------------------------
nodes_loaded = {
    4: (0.0, -400.0, 0.0)
}

dof_restrained_1based = restrained_dofs_1based(nodes_restrained, node_dofs_1based)
dof_loaded_1based = loaded_dofs_1based(nodes_loaded, node_dofs_1based)


### Q3.2 Analysis

Solve the truss using the space truss functions you need to develop (edit the code at the start of this workbook).

Report key numerical results, such as:
- Support reactions
- Member axial forces
- Maximum nodal displacements
- Present summary tables
- Plot displaced shape at a reasonable scale

In [199]:
print_toggle = 0
K_global = assemble_global_stiffness(elements, nodes, elements_lmnL, print_toggle)
f_global = assemble_global_force_vector(nodes, dof_loaded_1based)

(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    free_dofs,
    restrained_dofs,
) = partition_system(K_global, f_global, dof_restrained_1based)

u_f = solve_free_displacements(K_ff, K_fr, f_f)
F_r = solve_support_forces(K_rf, K_rr, u_f)
u_global = assemble_global_displacements(u_f, free_dofs, restrained_dofs)
f_global_complete = assemble_global_forces(f_f, F_r, free_dofs, restrained_dofs)


In [200]:
plot_truss_deformation(nodes, elements, u_global, scale=100)

In [201]:
# Plotting function for 3D trusses (Plotly version).
# This mirrors the Matplotlib, but works interactively everywhere (Jupyter, Colab, VS Code, etc.).

def plot_truss_3d_plotly(
    nodes,
    elements,
    nodes_restrained=None,
    nodes_loaded=None,
    show_node_ids=True,
    show_member_ids=False,
    load_scale=0.02,
):
    """
    nodes: {nid: (x,y,z)}
    elements: {eid: (i,j,E,A)}
    nodes_restrained: {nid: ["ux","uy","uz"], ...}
    nodes_loaded: {nid: (Fx,Fy,Fz)}
    """

    # ---- normalize nodes to 3D ----
    nodes3 = {}
    for nid, xyz in nodes.items():
        if len(xyz) == 2:
            x, y = xyz
            z = 0.0
        else:
            x, y, z = xyz
        nodes3[nid] = (float(x), float(y), float(z))

    fig = go.Figure()

    # =========================================================
    # Members (lines)
    # =========================================================
    xs, ys, zs = [], [], []
    for eid, (i, j, E, A) in elements.items():
        xi, yi, zi = nodes3[i]
        xj, yj, zj = nodes3[j]
        xs += [xi, xj, None]
        ys += [yi, yj, None]
        zs += [zi, zj, None]

    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="lines",
            line=dict(width=3),
            name="Members",
        )
    )

    # Optional member IDs (text at midpoints)
    if show_member_ids:
        mx, my, mz, mt = [], [], [], []
        for eid, (i, j, E, A) in elements.items():
            xi, yi, zi = nodes3[i]
            xj, yj, zj = nodes3[j]
            mx.append((xi + xj) / 2)
            my.append((yi + yj) / 2)
            mz.append((zi + zj) / 2)
            mt.append(str(eid))

        fig.add_trace(
            go.Scatter3d(
                x=mx,
                y=my,
                z=mz,
                mode="text",
                text=mt,
                name="Member IDs",
            )
        )

    # =========================================================
    # Nodes
    # =========================================================
    nx = [p[0] for p in nodes3.values()]
    ny = [p[1] for p in nodes3.values()]
    nz = [p[2] for p in nodes3.values()]

    fig.add_trace(
        go.Scatter3d(
            x=nx,
            y=ny,
            z=nz,
            mode="markers",
            marker=dict(size=4),
            name="Nodes",
        )
    )

    if show_node_ids:
        fig.add_trace(
            go.Scatter3d(
                x=nx,
                y=ny,
                z=nz,
                mode="text",
                text=[str(nid) for nid in nodes3.keys()],
                textposition="top center",
                name="Node IDs",
            )
        )

    # =========================================================
    # Supports
    # =========================================================
    if nodes_restrained:
        sx, sy, sz = [], [], []
        for nid in nodes_restrained.keys():
            x, y, z = nodes3[nid]
            sx.append(x)
            sy.append(y)
            sz.append(z)

        fig.add_trace(
            go.Scatter3d(
                x=sx,
                y=sy,
                z=sz,
                mode="markers",
                marker=dict(size=7),
                name="Supports",
            )
        )

    # =========================================================
    # Loads (true arrows using cones)
    # =========================================================
    if nodes_loaded:
        lx, ly, lz = [], [], []
        u, v, w = [], [], []

        for nid, (Fx, Fy, Fz) in nodes_loaded.items():
            x, y, z = nodes3[nid]
            lx.append(x)
            ly.append(y)
            lz.append(z)

            # direction + magnitude
            u.append(load_scale * Fx)
            v.append(load_scale * Fy)
            w.append(load_scale * Fz)

        fig.add_trace(
            go.Cone(
                x=lx,
                y=ly,
                z=lz,
                u=u,
                v=v,
                w=w,
                anchor="tail",   # arrow starts at node
                sizemode="absolute",
                sizeref=0.2,     # adjust arrowhead size
                showscale=False,
                name="Loads",
            )
        )


    # =========================================================
    # Layout (mirrors Matplotlib intent)
    # =========================================================
    fig.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode="data",  # equal-ish aspect
            camera=dict(
                eye=dict(x=1.4, y=1.4, z=0.9)  # similar to elev=20, azim=30
            ),
        ),
        margin=dict(l=0, r=0, t=30, b=0),
        title="3D Truss",
        showlegend=True,
    )

    fig.show()


In [202]:
plot_truss_3d_plotly(
    nodes,
    elements,
    nodes_restrained,
    nodes_loaded,
    show_node_ids=True,
    show_member_ids=False,
    load_scale=0.015,
)

In [203]:
results = recover_element_results(elements, elements_lmnL, u_global)
df_members = build_element_results_dataframe(elements, elements_lmnL, results)
display_compact(df_members)

,ele,i,j,L (mm),N (kN),sigma (MPa),u_1 (mm),u_2 (mm),u_3 (mm),u_4 (mm),u_5 (mm),u_6 (mm),u_1' (mm),u_2' (mm),u_3' (mm),u_4' (mm),u_5' (mm),u_6' (mm),f_1' (kN),f_2' (kN),f_3' (kN),f_4' (kN),f_5' (kN),f_6' (kN)
0,1,1,5,6000.0,-45.1,-11.3,0.0,0.0,0.0,-1.0,6.7,-6.4,0.0,0.0,0.0,-1.0,6.7,-6.4,45.1,0.0,0.0,-45.1,0.0,0.0
1,2,1,3,5196.2,33.8,8.4,0.0,0.0,0.0,-2.6,5.7,5.3,0.0,0.0,0.0,0.6,6.3,5.3,-33.8,0.0,0.0,33.8,0.0,0.0
2,3,3,5,3000.0,-0.0,-0.0,-2.6,5.7,5.3,-1.0,6.7,-6.4,-6.3,0.6,5.3,-6.3,2.5,-6.4,0.0,0.0,0.0,0.0,0.0,0.0
3,4,3,7,5196.2,42.3,10.6,-2.6,5.7,5.3,-2.9,7.8,24.2,0.6,6.3,5.3,1.4,8.2,24.2,-42.3,0.0,0.0,42.3,0.0,0.0
4,5,5,7,6000.0,0.0,0.0,-1.0,6.7,-6.4,-2.9,7.8,24.2,5.3,4.2,-6.4,5.3,6.4,24.2,-0.0,0.0,0.0,0.0,0.0,0.0
5,6,5,9,6000.0,-30.9,-7.7,-1.0,6.7,-6.4,-1.6,8.5,-25.7,-1.0,6.7,-6.4,-1.6,8.5,-25.7,30.9,0.0,0.0,-30.9,0.0,0.0
6,7,7,9,6000.0,0.0,0.0,-2.9,7.8,24.2,-1.6,8.5,-25.7,-8.2,1.4,24.2,-8.2,2.8,-25.7,-0.0,0.0,0.0,0.0,0.0,0.0
7,8,7,11,5196.2,58.8,14.7,-2.9,7.8,24.2,-1.3,8.3,-20.3,-6.4,5.3,24.2,-5.3,6.5,-20.3,-58.8,0.0,0.0,58.8,0.0,0.0
8,9,9,11,3000.0,0.0,0.0,-1.6,8.5,-25.7,-1.3,8.3,-20.3,6.5,5.7,-25.7,6.5,5.3,-20.3,0.0,0.0,0.0,0.0,0.0,0.0
9,10,9,13,6000.0,-94.3,-23.6,-1.6,8.5,-25.7,-3.6,0.0,-54.9,-1.6,8.5,-25.7,-3.6,0.0,-54.9,94.3,0.0,0.0,-94.3,0.0,0.0


In [204]:
R_df = build_reaction_df(
    nodes_restrained=nodes_restrained,
    node_dofs_1based=node_dofs_1based,
    K_global=K_global,
    u_global=u_global,
    dof_loaded_1based=dof_loaded_1based,
    decimals=3
)

display_compact(R_df)

,node,Rx (kN),Ry (kN),Rz (kN)
0,1,0.0,51.7,11.9
1,2,-0.0,248.3,-11.9
2,13,0.0,-51.7,0.0
3,14,-0.0,151.7,-0.0


In [205]:
df_disp = build_node_displacement_df(nodes, node_dofs_1based, u_global)

df_umax = build_max_displacement_summary(df_disp)

display_compact(df_umax)

,metric,node,value (mm),ux (mm),uy (mm),uz (mm)
0,max |u|,14,56.9,14.9,0.0,-54.9
1,max |ux|,4,16.5,16.5,-43.8,6.4
2,max |uy|,6,-44.7,7.6,-44.7,-7.2
3,max |uz|,13,-54.9,-3.6,0.0,-54.9


In [206]:
# Choose one element to sanity-check
e_test = 1

# --- Geometry + properties ---
i, j, E, A = elements[e_test]
l, m, n, L = elements_lmnL.get(e_test, element_lmnL(nodes[i], nodes[j]))

print(f"Element {e_test}: nodes ({i}, {j})")
print(f"l = {l:.6f}, m = {m:.6f}, n = {n:.6f}, L = {L:.3f} mm")

# --- Displacements ---
u_e = extract_element_displacements(u_global, i, j)
print(f"u_e (ele {e_test} global displacements) = {u_e}")

u_local = compute_local_displacements(l, m, n, u_e)
print(f"u_local (ele {e_test} local displacements) = {u_local}")

# --- Forces / stress ---
f_local = compute_local_end_forces(E, A, L, u_local)
print(f"f_local (ele {e_test} local end forces) = {f_local}")

N, sigma = compute_axial_force_and_stress(E, A, L, u_local)
print(f"Axial force  N = {N:.3f} kN")
print(f"Axial stress σ = {sigma:.6f} GPa  ({sigma*1000:.2f} MPa)")

f_global_e = local_to_global_forces(l, m, n, f_local)
print(f"f_global_e (ele {e_test} end forces in global coords) = {f_global_e}")


Element 1: nodes (1, 5)
l = 1.000000, m = 0.000000, n = 0.000000, L = 6000.000 mm
u_e (ele 1 global displacements) = [ 0.     0.     0.    -0.967  6.685 -6.423]
u_local (ele 1 local displacements) = [ 0.     0.     0.    -0.967  6.685 -6.423]
f_local (ele 1 local end forces) = [ 45.106   0.      0.    -45.106   0.      0.   ]
Axial force  N = -45.106 kN
Axial stress σ = -0.011277 GPa  (-11.28 MPa)
f_global_e (ele 1 end forces in global coords) = [ 45.106   0.      0.    -45.106   0.      0.   ]


### YOUR ANSWER FOR Q3.2 IN THIS CELL:

Use as many cells as you need

---

### Q3.3 Discussion

Compare the support reactions and member foces obtained from your code with those calculated by hand and with the 2D analysis performed in Q1. State whether they agree, and briefly summarize your findings in the final cell using **Markdown** and **LaTeX** as appropriate.

### YOUR ANSWER FOR Q1.3 IN THIS CELL:

Use as many cells as you need

---
---

## Question 4 — Case Study Space Truss Analysis (30 points)

Analyze the following space truss given your 3D code that you have verified gives correct answers from Q3. 

Pin-jointed **3D space truss** with **translational DOFs only** at each node:
$$(U_x, U_y, U_z)$$

Information given in units 
- Length: m
- Force: kN

### Model Summary

| Quantity | Value |
|--------:|------:|
| Nodes | 18 |
| Members | 53 |
| Total DOFs | $18 \times 3 = 54$ |
| Restrained DOFs | 6 |
| Free DOFs | 48 |
| Structural Type | 3D space truss |
| Load Case | Vertical + lateral |

$ E = 200 $ GPa

>Careful of units, make sure to convert to same basis


### Nodes

Bottom Layer ($z = 0$)

| Node | $x$ | $y$ | $z$ |
|-----:|----:|----:|----:|
| 1 | 0 | 0 | 0 |
| 2 | 1 | 0 | 0 |
| 3 | 2 | 0 | 0 |
| 4 | 0 | 1 | 0 |
| 5 | 1 | 1 | 0 |
| 6 | 2 | 1 | 0 |
| 7 | 0 | 2 | 0 |
| 8 | 1 | 2 | 0 |
| 9 | 2 | 2 | 0 |

Top Layer ($z = 1.2$)

| Node | $x$ | $y$ | $z$ |
|-----:|----:|----:|----:|
| 10 | 0 | 0 | 1.2 |
| 11 | 1 | 0 | 1.2 |
| 12 | 2 | 0 | 1.2 |
| 13 | 0 | 1 | 1.2 |
| 14 | 1 | 1 | 1.2 |
| 15 | 2 | 1 | 1.2 |
| 16 | 0 | 2 | 1.2 |
| 17 | 1 | 2 | 1.2 |
| 18 | 2 | 2 | 1.2 |


### Members (Connectivity)

Bottom Chord – Edges

| Member | $i$ | $j$ | $A$ ($mm^2$) |
|-------:|----:|----:|-------------:|
| 1 | 1 | 2 | 1200 |
| 2 | 2 | 3 | 1200 |
| 3 | 4 | 5 | 1200 |
| 4 | 5 | 6 | 1200 |
| 5 | 7 | 8 | 1200 |
| 6 | 8 | 9 | 1200 |
| 7 | 1 | 4 | 1200 |
| 8 | 4 | 7 | 1200 |
| 9 | 2 | 5 | 1200 |
| 10 | 5 | 8 | 1200 |
| 11 | 3 | 6 | 1200 |
| 12 | 6 | 9 | 1200 |

Bottom Chord – Diagonals

| Member | $i$ | $j$ | $A$ ($mm^2$) |
|-------:|----:|----:|-------------:|
| 13 | 1 | 5 | 900 |
| 14 | 2 | 6 | 900 |
| 15 | 4 | 8 | 900 |
| 16 | 5 | 9 | 900 |

Top Chord – Edges


| Member | $i$ | $j$ | $A$ ($mm^2$) |
|-------:|----:|----:|-------------:|
| 17 | 10 | 11 | 900 |
| 18 | 11 | 12 | 900 |
| 19 | 13 | 14 | 900 |
| 20 | 14 | 15 | 900 |
| 21 | 16 | 17 | 900 |
| 22 | 17 | 18 | 900 |
| 23 | 10 | 13 | 900 |
| 24 | 13 | 16 | 900 |
| 25 | 11 | 14 | 900 |
| 26 | 14 | 17 | 900 |
| 27 | 12 | 15 | 900 |
| 28 | 15 | 18 | 900 |

Top Chord – Diagonals

| Member | $i$ | $j$ | $A$ ($mm^2$) |
|-------:|----:|----:|-------------:|
| 29 | 10 | 14 | 700 |
| 30 | 11 | 15 | 700 |
| 31 | 13 | 17 | 700 |
| 32 | 14 | 18 | 700 |

Web Members - Verticals

| Member | $i$ | $j$ | $A$ ($mm^2$) |
|-------:|----:|----:|-------------:|
| 33 | 1 | 10 | 700 |
| 34 | 2 | 11 | 700 |
| 35 | 3 | 12 | 700 |
| 36 | 4 | 13 | 700 |
| 37 | 5 | 14 | 700 |
| 38 | 6 | 15 | 700 |
| 39 | 7 | 16 | 700 |
| 40 | 8 | 17 | 700 |
| 41 | 9 | 18 | 700 |


Web Members - Cross-Layer Diagonals

| Member | $i$ | $j$ | $A$ ($mm^2$) |
|-------:|----:|----:|-------------:|
| 42 | 10 | 2 | 600 |
| 43 | 11 | 3 | 600 |
| 44 | 13 | 5 | 600 |
| 45 | 14 | 6 | 600 |
| 46 | 16 | 8 | 600 |
| 47 | 17 | 9 | 600 |
| 48 | 10 | 4 | 600 |
| 49 | 13 | 7 | 600 |
| 50 | 11 | 5 | 600 |
| 51 | 14 | 8 | 600 |
| 52 | 12 | 6 | 600 |
| 53 | 15 | 9 | 600 |



### Supports / Fixities

Translational restraints only for truss.

| Node | $U_x$ | $U_y$ | $U_z$ |
|-----:|:----:|:----:|:----:|
| 1 | fixed | fixed | fixed |
| 3 | free | fixed | fixed |
| 7 | free | free | fixed |

Total restrained DOFs: $3 + 2 + 1 = 6$

### Applied Loads

Nodal Loads (kN)

| Node | $F_x$ | $F_y$ | $F_z$ |
|-----:|-----:|-----:|-----:|
| 10 | $+10$ | $+10$ | $-10$ |
| 11 | $+10$ | $+10$ | $-10$ |
| 12 | $+10$ | $+10$ | $-10$ |
| 13 | $+5$ | $0$ | $-10$ |
| 14 | $+5$ | $0$ | $-50$ |
| 15 | $+5$ | $0$ | $-10$ |
| 16 | $+5$ | $0$ | $-10$ |
| 17 | $+5$ | $0$ | $-10$ |
| 18 | $+5$ | $0$ | $-10$ |

All other nodes: $F_x = F_y = F_z = 0$


---

### Q4.1. Problem Setup
Define all model inputs (nodes, elements, supports, loads) using the same data structures and conventions as in **Question 3**.

Present:
- Node coordinates
- Element connectivity and properties
- Applied loads and boundary conditions

In [207]:
# -------------------------
# Nodes: (x,y,z)
# -------------------------
nodes = {
    # Bottom Layer (z = 0)
    1:  (0.0, 0.0, 0.0),
    2:  (1.0, 0.0, 0.0),
    3:  (2.0, 0.0, 0.0),
    4:  (0.0, 1.0, 0.0),
    5:  (1.0, 1.0, 0.0),
    6:  (2.0, 1.0, 0.0),
    7:  (0.0, 2.0, 0.0),
    8:  (1.0, 2.0, 0.0),
    9:  (2.0, 2.0, 0.0),

    # Top Layer (z = 1.2)
    10: (0.0, 0.0, 1.2),
    11: (1.0, 0.0, 1.2),
    12: (2.0, 0.0, 1.2),
    13: (0.0, 1.0, 1.2),
    14: (1.0, 1.0, 1.2),
    15: (2.0, 1.0, 1.2),
    16: (0.0, 2.0, 1.2),
    17: (1.0, 2.0, 1.2),
    18: (2.0, 2.0, 1.2)
}

In [208]:
E = 200 # GPa
A1 = 1200 # mm2
A2 = 900  # mm2
A3 = 700  # mm2
A4 = 600  # mm2

# -------------------------
# Elements: (i, j, E, A)
# -------------------------
elements = {
    # Bottom Chord – Edges
    1:  (1, 2, E, A1),
    2:  (2, 3, E, A1),
    3:  (4, 5, E, A1),
    4:  (5, 6, E, A1),
    5:  (7, 8, E, A1),
    6:  (8, 9, E, A1),
    7:  (1, 4, E, A1),
    8:  (4, 7, E, A1),
    9:  (2, 5, E, A1),
    10: (5, 8, E, A1),
    11: (3, 6, E, A1),
    12: (6, 9, E, A1),

    # Bottom Chord – Diagonals
    13: (1, 5, E, A2),
    14: (2, 6, E, A2),
    15: (4, 8, E, A2),
    16: (5, 9, E, A2),

    # Top Chord – Edges
    17: (10, 11, E, A2),
    18: (11, 12, E, A2),
    19: (13, 14, E, A2),
    20: (14, 15, E, A2),
    21: (16, 17, E, A2),
    22: (17, 18, E, A2),
    23: (10, 13, E, A2),
    24: (13, 16, E, A2),
    25: (11, 14, E, A2),
    26: (14, 17, E, A2),
    27: (12, 15, E, A2),
    28: (15, 18, E, A2),

    # Top Chord – Diagonals
    29: (10, 14, E, A3),
    30: (11, 15, E, A3),
    31: (13, 17, E, A3),
    32: (14, 18, E, A3),

    # Web Members – Verticals
    33: (1, 10, E, A3),
    34: (2, 11, E, A3),
    35: (3, 12, E, A3),
    36: (4, 13, E, A3),
    37: (5, 14, E, A3),
    38: (6, 15, E, A3),
    39: (7, 16, E, A3),
    40: (8, 17, E, A3),
    41: (9, 18, E, A3),

    # Web Members – Diagonals
    42: (10, 2, E, A4),
    43: (11, 3, E, A4),
    44: (13, 5, E, A4),
    45: (14, 6, E, A4),
    46: (16, 8, E, A4),
    47: (17, 9, E, A4),
    48: (10, 4, E, A4),
    49: (13, 7, E, A4),
    50: (11, 5, E, A4),
    51: (14, 8, E, A4),
    52: (12, 6, E, A4),
    53: (15, 9, E, A4)
}

elements_lmnL = build_elements_lmnL(elements, nodes)

In [209]:
# -------------------------
# Supports / restraints
# -------------------------
nodes_restrained = {
    1: ["ux", "uy", "uz"],
    3: ["uy", "uz"],
    7: ["uz"]
}

# -------------------------
# Applied nodal loads (Fx, Fy, Fz)
# -------------------------
nodes_loaded = {
    10: (10.0, 10.0, -10.0),
    11: (10.0, 10.0, -10.0),
    12: (10.0, 10.0, -10.0),
    13: (5.0,  0.0,  -10.0),
    14: (5.0,  0.0,  -50.0),
    15: (5.0,  0.0,  -10.0),
    16: (5.0,  0.0,  -10.0),
    17: (5.0,  0.0,  -10.0),
    18: (5.0,  0.0,  -10.0)
}

dof_restrained_1based = restrained_dofs_1based(nodes_restrained, node_dofs_1based)
dof_loaded_1based = loaded_dofs_1based(nodes_loaded, node_dofs_1based)

In [210]:
# Plotting function for 3D trusses (Plotly version).
# This mirrors the Matplotlib, but works interactively everywhere (Jupyter, Colab, VS Code, etc.).

def plot_truss_3d_plotly(
    nodes,
    elements,
    nodes_restrained=None,
    nodes_loaded=None,
    show_node_ids=True,
    show_member_ids=False,
    load_scale=0.02,
):
    """
    nodes: {nid: (x,y,z)}
    elements: {eid: (i,j,E,A)}
    nodes_restrained: {nid: ["ux","uy","uz"], ...}
    nodes_loaded: {nid: (Fx,Fy,Fz)}
    """

    # ---- normalize nodes to 3D ----
    nodes3 = {}
    for nid, xyz in nodes.items():
        if len(xyz) == 2:
            x, y = xyz
            z = 0.0
        else:
            x, y, z = xyz
        nodes3[nid] = (float(x), float(y), float(z))

    fig = go.Figure()

    # =========================================================
    # Members (lines)
    # =========================================================
    xs, ys, zs = [], [], []
    for eid, (i, j, E, A) in elements.items():
        xi, yi, zi = nodes3[i]
        xj, yj, zj = nodes3[j]
        xs += [xi, xj, None]
        ys += [yi, yj, None]
        zs += [zi, zj, None]

    fig.add_trace(
        go.Scatter3d(
            x=xs,
            y=ys,
            z=zs,
            mode="lines",
            line=dict(width=3),
            name="Members",
        )
    )

    # Optional member IDs (text at midpoints)
    if show_member_ids:
        mx, my, mz, mt = [], [], [], []
        for eid, (i, j, E, A) in elements.items():
            xi, yi, zi = nodes3[i]
            xj, yj, zj = nodes3[j]
            mx.append((xi + xj) / 2)
            my.append((yi + yj) / 2)
            mz.append((zi + zj) / 2)
            mt.append(str(eid))

        fig.add_trace(
            go.Scatter3d(
                x=mx,
                y=my,
                z=mz,
                mode="text",
                text=mt,
                name="Member IDs",
            )
        )

    # =========================================================
    # Nodes
    # =========================================================
    nx = [p[0] for p in nodes3.values()]
    ny = [p[1] for p in nodes3.values()]
    nz = [p[2] for p in nodes3.values()]

    fig.add_trace(
        go.Scatter3d(
            x=nx,
            y=ny,
            z=nz,
            mode="markers",
            marker=dict(size=4),
            name="Nodes",
        )
    )

    if show_node_ids:
        fig.add_trace(
            go.Scatter3d(
                x=nx,
                y=ny,
                z=nz,
                mode="text",
                text=[str(nid) for nid in nodes3.keys()],
                textposition="top center",
                name="Node IDs",
            )
        )

    # =========================================================
    # Supports
    # =========================================================
    if nodes_restrained:
        sx, sy, sz = [], [], []
        for nid in nodes_restrained.keys():
            x, y, z = nodes3[nid]
            sx.append(x)
            sy.append(y)
            sz.append(z)

        fig.add_trace(
            go.Scatter3d(
                x=sx,
                y=sy,
                z=sz,
                mode="markers",
                marker=dict(size=7),
                name="Supports",
            )
        )

    # =========================================================
    # Loads (true arrows using cones)
    # =========================================================
    if nodes_loaded:
        lx, ly, lz = [], [], []
        u, v, w = [], [], []

        for nid, (Fx, Fy, Fz) in nodes_loaded.items():
            x, y, z = nodes3[nid]
            lx.append(x)
            ly.append(y)
            lz.append(z)

            # direction + magnitude
            u.append(load_scale * Fx)
            v.append(load_scale * Fy)
            w.append(load_scale * Fz)

        fig.add_trace(
            go.Cone(
                x=lx,
                y=ly,
                z=lz,
                u=u,
                v=v,
                w=w,
                anchor="tail",   # arrow starts at node
                sizemode="absolute",
                sizeref=0.2,     # adjust arrowhead size
                showscale=False,
                name="Loads",
            )
        )


    # =========================================================
    # Layout (mirrors Matplotlib intent)
    # =========================================================
    fig.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode="data",  # equal-ish aspect
            camera=dict(
                eye=dict(x=1.4, y=1.4, z=0.9)  # similar to elev=20, azim=30
            ),
        ),
        margin=dict(l=0, r=0, t=30, b=0),
        title="3D Truss",
        showlegend=True,
    )

    fig.show()


In [211]:
plot_truss_3d_plotly(
    nodes,
    elements,
    nodes_restrained,
    nodes_loaded,
    show_node_ids=True,
    show_member_ids=False,
    load_scale=0.015,
)


---

### Q4.2. Analysis

Solve the truss using the space truss functions you need to developed in Q3.

Report key numerical results, such as:
- Support reactions
- Member axial forces
- Maximum nodal displacements
- Present summary tables
- Plot displaced shape at a reasonable scale

In [212]:
print_toggle = 0
K_global = assemble_global_stiffness(elements, nodes, elements_lmnL, print_toggle)
f_global = assemble_global_force_vector(nodes, dof_loaded_1based)

(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    free_dofs,
    restrained_dofs,
) = partition_system(K_global, f_global, dof_restrained_1based)

u_f = solve_free_displacements(K_ff, K_fr, f_f)
F_r = solve_support_forces(K_rf, K_rr, u_f)
u_global = assemble_global_displacements(u_f, free_dofs, restrained_dofs)
f_global_complete = assemble_global_forces(f_f, F_r, free_dofs, restrained_dofs)


In [213]:
plot_truss_deformation(nodes, elements, u_global, scale=100)

In [214]:
results = recover_element_results(elements, elements_lmnL, u_global)
df_members = build_element_results_dataframe(elements, elements_lmnL, results)
display_compact(df_members)

,ele,i,j,L (mm),N (kN),sigma (MPa),u_1 (mm),u_2 (mm),u_3 (mm),u_4 (mm),u_5 (mm),u_6 (mm),u_1' (mm),u_2' (mm),u_3' (mm),u_4' (mm),u_5' (mm),u_6' (mm),f_1' (kN),f_2' (kN),f_3' (kN),f_4' (kN),f_5' (kN),f_6' (kN)
0,1,1,2,1.0,64.7,53.9,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,-64.7,0.0,0.0,64.7,0.0,0.0
1,2,2,3,1.0,46.7,38.9,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,-46.7,0.0,0.0,46.7,0.0,0.0
2,3,4,5,1.0,16.1,13.4,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-16.1,0.0,0.0,16.1,0.0,0.0
3,4,5,6,1.0,28.9,24.1,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,-28.9,0.0,0.0,28.9,0.0,0.0
4,5,7,8,1.0,0.0,0.0,-0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,6,8,9,1.0,9.4,7.9,-0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,-9.4,0.0,0.0,9.4,0.0,0.0
6,7,1,4,1.0,42.2,35.2,0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,-42.2,0.0,0.0,42.2,0.0,0.0
7,8,4,7,1.0,35.3,29.4,-0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,-35.3,0.0,0.0,35.3,0.0,0.0
8,9,2,5,1.0,12.4,10.3,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-12.4,0.0,0.0,12.4,0.0,0.0
9,10,5,8,1.0,29.3,24.4,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0,-0.0,-29.3,0.0,0.0,29.3,0.0,0.0


In [215]:
R_df = build_reaction_df(
    nodes_restrained=nodes_restrained,
    node_dofs_1based=node_dofs_1based,
    K_global=K_global,
    u_global=u_global,
    dof_loaded_1based=dof_loaded_1based,
    decimals=3
)

display_compact(R_df)

,node,Rx (kN),Ry (kN),Rz (kN)
0,1,-60.0,-37.5,-54.0
1,3,0.0,7.5,101.0
2,7,0.0,-0.0,83.0


In [216]:
df_disp = build_node_displacement_df(nodes, node_dofs_1based, u_global)

df_umax = build_max_displacement_summary(df_disp)

display_compact(df_umax)

,metric,node,value (mm),ux (mm),uy (mm),uz (mm)
0,max |u|,1,0.0,0.0,0.0,0.0
1,max |ux|,1,0.0,0.0,0.0,0.0
2,max |uy|,1,0.0,0.0,0.0,0.0
3,max |uz|,1,0.0,0.0,0.0,0.0


In [217]:
# Choose one element to sanity-check
e_test = 1

# --- Geometry + properties ---
i, j, E, A = elements[e_test]
l, m, n, L = elements_lmnL.get(e_test, element_lmnL(nodes[i], nodes[j]))

print(f"Element {e_test}: nodes ({i}, {j})")
print(f"l = {l:.6f}, m = {m:.6f}, n = {n:.6f}, L = {L:.3f} mm")

# --- Displacements ---
u_e = extract_element_displacements(u_global, i, j)
print(f"u_e (ele {e_test} global displacements) = {u_e}")

u_local = compute_local_displacements(l, m, n, u_e)
print(f"u_local (ele {e_test} local displacements) = {u_local}")

# --- Forces / stress ---
f_local = compute_local_end_forces(E, A, L, u_local)
print(f"f_local (ele {e_test} local end forces) = {f_local}")

N, sigma = compute_axial_force_and_stress(E, A, L, u_local)
print(f"Axial force  N = {N:.3f} kN")
print(f"Axial stress σ = {sigma:.6f} GPa  ({sigma*1000:.2f} MPa)")

f_global_e = local_to_global_forces(l, m, n, f_local)
print(f"f_global_e (ele {e_test} end forces in global coords) = {f_global_e}")


Element 1: nodes (1, 2)
l = 1.000000, m = 0.000000, n = 0.000000, L = 1.000 mm
u_e (ele 1 global displacements) = [ 0.  0.  0.  0. -0. -0.]
u_local (ele 1 local displacements) = [ 0.  0.  0.  0. -0. -0.]
f_local (ele 1 local end forces) = [-64.68   0.     0.    64.68   0.     0.  ]
Axial force  N = 64.680 kN
Axial stress σ = 0.053900 GPa  (53.90 MPa)
f_global_e (ele 1 end forces in global coords) = [-64.68   0.     0.    64.68   0.     0.  ]


### YOUR ANSWER FOR Q4.2 IN THIS CELL:

Use as many cells as you need

---


### Q4.3. Discussion
Discuss the computed **member forces** and **nodal displacements**.

Interpret your results from an engineering perspective by addressing the following questions:

- Are the computed displacements and deformed shape physically reasonable for the structure and loading considered?
- Do the signs and magnitudes of member forces and support reactions make sense?
- Which members carry the largest axial forces, and how does this relate to the expected load paths?
- How do your modeling assumptions (geometry, cross-sections, boundary conditions) influence the results?
- Which assumptions do you expect the results to be most sensitive to, and why?


### YOUR ANSWER FOR Q4.3 IN THIS CELL:

Use as many cells as you need

#### Are the computed displacements and deformed shape physically reasonable for the structure and loading considered?

- Yes, they are.

#### Do the signs and magnitudes of member forces and support reactions make sense?

- Yes.

#### Which members carry the largest axial forces, and how does this relate to the expected load paths?

The members experiencing the largest axial forces are primarily concentrated along the outer contour of the spatial truss, specifically the upper and lower chords of the two rows of longitudinal trusses, as well as the diagonal and vertical members in the web plates near the load points and supports. Their load transmission paths align with expectations. Vertical concentrated forces at nodes are first decomposed into axial forces within members surrounding the load node. Web members transmit shear forces from equivalent panels to chords, forming tension-compression couples. Simultaneously, chords continuously transfer internal forces along the span direction to supports, resulting in axial force peaks at contour chords and critical web members.


#### How do your modeling assumptions (geometry, cross-sections, boundary conditions) influence the results?

- The height of the truss and the length of the bay determine how much "equivalent beam bending moment" the chord members must bear, thus directly affecting the magnitude of axial force experienced by the members.
- The larger the cross-sectional area of a member, the greater its stiffness, and the more force it can bear.
- Applying additional constraints under statically determinate boundary conditions alters the load transfer path, thereby changing the stress distribution within the members.

#### Which assumptions do you expect the results to be most sensitive to, and why?


- The results are most sensitive to boundary conditions. Changes in boundary conditions typically alter the force transmission paths, thereby modifying the internal force distribution within the structure.
- The results are also highly sensitive to member stiffness. For statically indeterminate structures, altering the cross-sectional area affects the overall stiffness of the structure, which in turn influences the distribution of internal forces.

---

## Reflection (Required)

In 3–6 sentences:

- What part felt easiest? Describe in speciics
- What part felt hardest? Describe in speciics
- One bug you encountered and how you fixed it.
- One thing you still don’t understand.
- If you used collaboration or AI tools, briefly describe how.


### YOUR RESPONSE HERE

#### What part felt easiest? Describe in speciics

- None.

#### What part felt hardest? Describe in speciics

- Of course the part where the code should be changed from 2D to 3D. When modifying code, it's easy to overlook statements involving loop control and dictionary item references.

#### One bug you encountered and how you fixed it.

- I was constantly told the matrix was singular, until I finally discovered that the loop control element in the assemble_global_stiffness function hadn't been changed from 4 to 6. I found this by printing the stiffness matrix.

#### One thing you still don’t understand.

- None.

#### If you used collaboration or AI tools, briefly describe how.

- I used ChatGPT for all the input of the elements and nodes.
- The ey and ez components of the truss_transformation_matrix function were also written with ChatGPT's assistance.
- ChatGPT also provided assistance in selecting boundary constraints during Q3.
- The bearing reaction force table was also written with ChatGPT's help.